#Query the Endpoint Directly
This cell calls Azure AI Foundry **directly** (bypassing Databricks Model Serving) using an AAD token obtained via service principal credentials stored in an Azure Key Vault-backed secret scope. The token is acquired against the `https://cognitiveservices.azure.com/.default` scope and passed to the `AzureOpenAI` client. <BR> The service principal called has the following RBAC role: **Cognitive Services OpenAI User**.

In [0]:
import requests
from openai import AzureOpenAI

key_vault_name = "DatabricksKV-JackBender"

# Acquire AAD token using service principal credentials
_tenant = dbutils.secrets.get(scope=key_vault_name, key="SPTenantID")
_token_resp = requests.post(
    f"https://login.microsoftonline.com/{_tenant}/oauth2/v2.0/token",
    data={
        "grant_type": "client_credentials",
        "client_id": dbutils.secrets.get(scope=key_vault_name, key="SPDatabricksAppID"),
        "client_secret": dbutils.secrets.get(scope=key_vault_name, key="SPDatabricksKey"),
        "scope": "https://cognitiveservices.azure.com/.default"
    }
)
_token_resp.raise_for_status()
_access_token = _token_resp.json()["access_token"]

# Call Azure AI Foundry directly (bypasses Model Serving Entra limitation with KV-backed scopes)
client = AzureOpenAI(
    azure_endpoint="https://<foundry project name>-resource.services.ai.azure.com",  # Foundry API Endpoint on the Foundry Resource
    azure_ad_token=_access_token,
    api_version="2024-12-01-preview"
)

response = client.chat.completions.create(
    model="gpt-5.6-luna-adbws",  # model name in Azure Foundry
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the population of New York City?"}
    ],
)

display(response.choices[0].message.content)